# LAB-03 | Auto Loader ingestion with schema evolution

## Acceptance Criteria: 
 - A streaming table ingests JSON from the landing volume, exactly-once
 - Each row carries source file name and ingestion timestamp
 - Adding the drift file causes the stream to fail, and a restart picks up the new column
 - You can state the default value of cloudFiles.schemaEvolutionMode and why it differs when a schema is supplied
 - You've confirmed _rescued_data behaviour with a deliberately mistyped value
 - A second run with no new files processes zero records

### Subtasks
 - Build the bronze streaming table — Hint: in SQL use STREAM read_files(...); in Python use spark.readStream.format("cloudFiles"). Inside a pipeline, checkpoints are managed for you.
 - Add file metadata columns — Hint: the _metadata column exposes file_name, file_size, file_modification_time.
 - Run and record the row count.
 - Drop in the drift file and re-run — Expect failure. Read the error text carefully before restarting.
 - Restart and confirm the new column appears.
 - Explain the default — Write your answer in a markdown cell, then check schema inference and evolution.
 - Test idempotency — Re-run with no new files. Zero rows means exactly-once is working.
 - Compare formats — Point a second stream at the CSV directory. Hint: JSON/CSV/XML infer everything as strings; Parquet and Avro carry types.

# Plan of Attack

## Setup
- [DONE] 0. Move batch_4 and batch_5 out of json/parquet landing dirs (drift trigger, added later)

## Ingest
- [DONE] 1. CSV → bronze via Auto Loader in a notebook (Python, manual checkpoint + schemaLocation)
- [UNDER DEVELOPMENT] 2. JSON & Parquet → two bronze streaming tables in one pipeline (SQL)
    - [TODO] Develop on Local Environment and deployes as a Pipeline

## Observe
3. Re-run both with no new files → expect 0 rows
4. Restore batch_4 → expect failure → restart → confirm fare_per_mile
5. Ingest the hand-written bad JSON → inspect _rescued_data
6. DESCRIBE json vs parquet bronze → compare inferred types

## Wrap
7. Job with a single pipeline task
8. Write-up: what the pipeline managed that I configured by hand

In [0]:
## DANGER ZONE: RUN ONLY IF NEED TO CLEAN UP CSV FILES, CHECKPOINTS AND TARGET TABLE
# base_checkpoint_location:   str     = "/Volumes/dev_ppetkov1/ops/landing/streaming_checkpoints/{format_type}"
# schema_checkpoint_location: str     = f"{base_checkpoint_location}/schema/"
# stream_checkpoint_location: str     = f"{base_checkpoint_location}/stream/"

# # 1. The table
# spark.sql("DROP TABLE IF EXISTS dev_ppetkov1.bronze.trips_csv")

# # 2. The checkpoint — tracks which files were processed
# dbutils.fs.rm(stream_checkpoint_location, recurse=True)

# # 3. The schema location — separate directory, easy to forget
# dbutils.fs.rm(schema_checkpoint_location, recurse=True)

# # 4. The landing files you're regenerating
# dbutils.fs.rm("/Volumes/dev_ppetkov1/ops/landing/nyctaxi_trips_csv/", recurse=True)

# 0. Move batch_4 and batch_5 out of json/parquet landing dirs (drift trigger, added later)

In [0]:
# [EXECUTED]
# Step 01: Setup
# Move batch_4 and batch_5 out of json/parquet landing dirs (drift trigger, added later)


file_types: list = ["csv", "json", "parquet"]

for format_type in file_types:
  for batch in ["batch_4", "batch_5"]:
    source_vol_address: str = f"/Volumes/dev_ppetkov1/ops/landing/nyctaxi_trips_{format_type}/{batch}"
    target_vol_address: str = f"/Volumes/dev_ppetkov1/ops/landing/schema_drift_n_dirty_records/{format_type}/{batch}"
    dbutils.fs.mv(source_vol_address, target_vol_address, True)

# CSV → bronze via Auto Loader in a notebook (Python, manual checkpoint + schemaLocation)

In [0]:
# Source dataset files
format_type:    str = "csv"
source_dir:     str = f"/Volumes/dev_ppetkov1/ops/landing/nyctaxi_trips_{format_type}/"
# Checkpoints
base_checkpoint_location:   str     = "/Volumes/dev_ppetkov1/ops/landing/streaming_checkpoints/{format_type}"
schema_checkpoint_location: str     = f"{base_checkpoint_location}/schema/"
stream_checkpoint_location: str     = f"{base_checkpoint_location}/stream/"

In [0]:
reader = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", format_type)
    .option("cloudFiles.schemaLocation", schema_checkpoint_location)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.schemaHints", "fare_amount double")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.includeExistingFiles", "true")
    .option("cloudFiles.allowOverwrites", "false")
    .option("rescuedDataColumn", "_rescued_data")
    .option("pathGlobFilter", f"part-*.{format_type}")
)

reader = (
    reader
    .option("header", "true")
    .option("delimiter", ",")
    .option("multiLine", "false")
    .option("nullValue", "")
)

bronze_df = (
    reader.load(source_dir)
    .selectExpr(
        "*",
        "_metadata.file_name  AS source_file",
        "_metadata.file_path  AS source_path",
        "_metadata.file_modification_time AS source_modified_at",
        "current_timestamp()  AS ingested_at",
    )
)

(
    bronze_df.writeStream
    .option("checkpointLocation", stream_checkpoint_location)
    .trigger(availableNow=True)
    .toTable(f"dev_ppetkov1.bronze.trips_{format_type}")
)

# Tables Analysis

In [0]:
%sql
describe extended dev_ppetkov1.bronze.trips_csv;

In [0]:
%sql
-- Returned 0 rows
select *
from dev_ppetkov1.bronze.trips_csv
where _rescued_data is not null;

In [0]:
%sql
select *
from dev_ppetkov1.bronze.trips_csv
order by 1
limit 10;


In [0]:
%sql
SELECT source_file, COUNT(*) AS rows, MIN(ingested_at) AS first_seen
FROM dev_ppetkov1.bronze.trips_csv
GROUP BY source_file ORDER BY source_file;

In [0]:
%sql
SELECT COUNT(*) FROM dev_ppetkov1.bronze.trips_csv;